In [1]:
import pandas as pd
import numpy as np

# Load merged dataset
df = pd.read_csv(
    "Delhi_AQI_2025_Merged.csv",
    na_values=["NA", "", " ", "null", "NULL"]
)

# ---------------------------------------------------
# 1. Clean Column Names
# ---------------------------------------------------
df.columns = (
    df.columns
      .str.replace("\ufeff", "", regex=False)
      .str.strip()
)

# Remove units from column names
df.columns = (
    df.columns
      .str.replace(" (µg/m³)", "", regex=False)
      .str.replace(" (mg/m³)", "", regex=False)
      .str.replace(" (°C)", "", regex=False)
      .str.replace(" (%)", "", regex=False)
      .str.replace(" (m/s)", "", regex=False)
      .str.replace(" (deg)", "", regex=False)
      .str.replace(" (mm)", "", regex=False)
      .str.replace(" (W/mt2)", "", regex=False)
      .str.replace(" (ppb)", "", regex=False)
      .str.replace(" (mmHg)", "", regex=False)
)

print(df.columns)

# ---------------------------------------------------
# 2. Convert Timestamp
# ---------------------------------------------------
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    dayfirst=True,
    errors="coerce"
)

# Remove invalid timestamps
df = df.dropna(subset=["Timestamp"])

# ---------------------------------------------------
# 3. Clean Station Name
# ---------------------------------------------------
df["Station"] = (
    df["Station"]
      .astype(str)
      .str.strip()
      .str.replace(",", "", regex=False)
)

# ---------------------------------------------------
# 4. Convert numeric columns
# ---------------------------------------------------
exclude = ["Timestamp", "Station"]

numeric_cols = [c for c in df.columns if c not in exclude]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------
# 5. Remove duplicate rows
# ---------------------------------------------------
df = df.drop_duplicates()

# ---------------------------------------------------
# 6. Remove rows having all pollution values missing
# ---------------------------------------------------
pollutants = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "SO2",
    "CO",
    "Ozone"
]

df = df.dropna(
    subset=pollutants,
    how="all"
)

# ---------------------------------------------------
# 7. Drop columns having >95% missing values
# ---------------------------------------------------
missing = df.isna().mean()

drop_cols = missing[missing > 0.95].index.tolist()

print("Dropping:", drop_cols)

df.drop(columns=drop_cols, inplace=True)

# ---------------------------------------------------
# 8. Fill weather values
# ---------------------------------------------------
weather_cols = [
    "AT",
    "RH",
    "WS",
    "WD",
    "RF",
    "TOT-RF",
    "SR",
    "BP",
    "VWS"
]

for c in weather_cols:
    if c in df.columns:
        df[c] = df[c].interpolate()

# ---------------------------------------------------
# 9. Sort
# ---------------------------------------------------
df = df.sort_values(
    ["Station", "Timestamp"]
)

# ---------------------------------------------------
# 10. Create Date Columns
# ---------------------------------------------------
df["Date"] = df["Timestamp"].dt.date
df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Month Name"] = df["Timestamp"].dt.strftime("%B")
df["Day"] = df["Timestamp"].dt.day
df["Hour"] = df["Timestamp"].dt.hour
df["Weekday"] = df["Timestamp"].dt.day_name()
df["Quarter"] = df["Timestamp"].dt.quarter

# ---------------------------------------------------
# 11. Save
# ---------------------------------------------------
df.to_csv(
    "Delhi_AQI_2025_Cleaned.csv",
    index=False
)

print("--------------------------------")
print("Cleaning Completed")
print("--------------------------------")
print(df.shape)
print(df.head())

C:\Users\Leo\AppData\Local\Temp\ipykernel_19200\1966012555.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Index(['Timestamp', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'SO2', 'CO',
       'Ozone', 'Benzene', 'Toluene', 'Xylene', 'O Xylene', 'Eth-Benzene',
       'MP-Xylene', 'AT', 'RH', 'WS', 'WD', 'RF', 'TOT-RF', 'SR', 'BP', 'VWS',
       'Station'],
      dtype='object')
Dropping: ['O Xylene']
--------------------------------
Cleaning Completed
--------------------------------
(134535, 33)
            Timestamp  PM2.5   PM10      NO    NO2     NOx    NH3    SO2  \
0 2025-01-01 00:00:00  251.0  374.0  102.03  58.83  114.27  74.83  13.20   
1 2025-01-01 01:00:00  269.0  437.0  102.10  61.60  115.80  74.32  12.35   
2 2025-01-01 02:00:00  238.0  326.0   59.90  63.22   82.35  81.50  14.50   
3 2025-01-01 03:00:00  227.0  324.0   44.08  61.25   68.40  70.03  16.45   
4 2025-01-01 04:00:00  286.0  381.0   31.90  51.35   53.25  66.12  15.75   

     CO  Ozone  ...  VWS  Station        Date  Year  Month  Month Name  Day  \
0  2.81   1.93  ...  NaN   Alipur  2025-01-01  2025      1     January 

In [1]:
import os

print(os.getcwd())

C:\Users\Leo\aqi


In [2]:
import pandas as pd
import numpy as np

# ============================
# Load Dataset
# ============================
file_path = "Delhi_AQI_2025_Cleaned.csv"   # Change if needed

df = pd.read_csv(file_path)

# ============================
# Convert Timestamp
# ============================
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# ============================
# Create Date Columns
# ============================
df["Date"] = df["Timestamp"].dt.date
df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Month Name"] = df["Timestamp"].dt.strftime("%B")
df["Day"] = df["Timestamp"].dt.day
df["Hour"] = df["Timestamp"].dt.hour
df["Weekday"] = df["Timestamp"].dt.day_name()
df["Quarter"] = df["Timestamp"].dt.quarter
df["Month-Year"] = df["Timestamp"].dt.strftime("%b-%Y")
df["Year-Month"] = df["Timestamp"].dt.to_period("M").astype(str)
df["Week Number"] = df["Timestamp"].dt.isocalendar().week

# ============================
# AQI (Temporary)
# Replace this with CPCB AQI later
# ============================

pollutants = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "Ozone",
    "NH3"
]

available = [c for c in pollutants if c in df.columns]

df["AQI"] = df[available].max(axis=1)

# ============================
# AQI Category
# ============================

def aqi_category(aqi):

    if pd.isna(aqi):
        return np.nan

    elif aqi <= 50:
        return "Good"

    elif aqi <= 100:
        return "Satisfactory"

    elif aqi <= 200:
        return "Moderate"

    elif aqi <= 300:
        return "Poor"

    elif aqi <= 400:
        return "Very Poor"

    else:
        return "Severe"

df["AQI Category"] = df["AQI"].apply(aqi_category)

# ============================
# Season
# ============================

def season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Summer"

    elif month in [6,7,8,9]:
        return "Monsoon"

    else:
        return "Post-Monsoon"

df["Season"] = df["Month"].apply(season)

# ============================
# Weekend / Weekday
# ============================

df["Day Type"] = np.where(
    df["Weekday"].isin(["Saturday","Sunday"]),
    "Weekend",
    "Weekday"
)

# ============================
# Day / Night
# ============================

df["Day/Night"] = np.where(
    (df["Hour"] >= 6) & (df["Hour"] < 18),
    "Day",
    "Night"
)

# ============================
# Pollution Level
# ============================

def pollution_level(aqi):

    if pd.isna(aqi):
        return np.nan

    elif aqi <= 100:
        return "Low"

    elif aqi <= 200:
        return "Medium"

    else:
        return "High"

df["Pollution Level"] = df["AQI"].apply(pollution_level)

# ============================
# Hour Category
# ============================

def hour_category(hour):

    if 0 <= hour < 6:
        return "Late Night"

    elif 6 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 18:
        return "Afternoon"

    else:
        return "Evening"

df["Hour Category"] = df["Hour"].apply(hour_category)

# ============================
# Save
# ============================

output_file = "Delhi_AQI_Final_Dashboard.csv"

df.to_csv(output_file, index=False)

print("="*50)
print("Dashboard Dataset Created Successfully")
print("="*50)
print("Rows :", len(df))
print("Columns :", len(df.columns))
print("Saved As :", output_file)

Dashboard Dataset Created Successfully
Rows : 134535
Columns : 43
Saved As : Delhi_AQI_Final_Dashboard.csv
